# Day 4—Measuring the shape of a sign
*Measuring Manuscripts*

A letter isn't one shape but a range of shapes; every scribe and every century draws it a little differently. Today we turn sign images into numbers and ask which signs are most alike—a teaching-scale version of what Isut's *Guesser* does. We'll use made-up signs so the whole pipeline runs; your real images go in at the end.

## 1. Setup

In [ ]:
!pip install scikit-image scikit-learn --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Some sign images to work with

Three archetypes—a cross, a ring, a diagonal—each with several noisy, slightly shifted copies, since no scribe draws the same sign twice.

In [ ]:
rng = np.random.default_rng(0)
S = 16  # image size

def cross():
    a = np.zeros((S, S)); a[S//2-1:S//2+1, :] = 1; a[:, S//2-1:S//2+1] = 1; return a
def ring():
    yy, xx = np.ogrid[:S, :S]; r = (yy - S/2)**2 + (xx - S/2)**2
    a = np.zeros((S, S)); a[(r < (S/2)**2) & (r > (S/2 - 3)**2)] = 1; return a
def diag():
    a = np.zeros((S, S))
    for k in range(S): a[k, min(S-1, k)] = 1; a[k, max(0, k-1)] = 1
    return a

archetypes = {'cross': cross, 'ring': ring, 'diag': diag}
images, labels = [], []
for name, fn in archetypes.items():
    for _ in range(5):
        img = fn() + 0.15 * rng.standard_normal((S, S))   # add noise
        img = np.clip(img, 0, 1)
        images.append(img); labels.append(name)
images = np.array(images)
print(images.shape[0], 'sign images,', S, 'x', S, 'pixels each')

## 3. Look at them

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(8, 5))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(img, cmap='gray_r'); ax.set_title(lab, fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. From image to numbers

To compare shapes, each image becomes a row of numbers—a **feature vector**. The simplest features are the pixels themselves, flattened into one row. Real systems use better features; that's the stub at the end.

In [ ]:
features = images.reshape(len(images), -1)   # 16x16 image -> 256 numbers
print('Feature matrix:', features.shape, '(one row per sign)')

## 5. Which signs are most similar?

Cosine similarity scores every pair from 0 (nothing in common) to 1 (identical). For each sign, which are its nearest neighbors?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(features)

def nearest(idx, k=3):
    order = np.argsort(-sim[idx])
    order = [j for j in order if j != idx][:k]
    return [(labels[j], round(float(sim[idx][j]), 2)) for j in order]

for idx in [0, 5, 10]:
    print(f'sign {idx} ({labels[idx]}) -> nearest:', nearest(idx))

## 6. A similarity heatmap

The whole comparison at once. Bright blocks along the diagonal mean each archetype's copies resemble each other more than the others, which is what we'd expect.

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(sim, cmap='viridis')
plt.colorbar(label='cosine similarity')
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.title('Every sign vs every sign'); plt.tight_layout(); plt.show()

## 7. Cluster the signs

Group the signs by shape without giving the computer the labels. If the clusters match the archetypes, the features are working. On real data the clusters might track scribes, periods, or genres instead.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
Z = linkage(features, method='ward')
plt.figure(figsize=(9, 3))
dendrogram(Z, labels=labels, leaf_rotation=90)
plt.title('Clustering signs by shape'); plt.tight_layout(); plt.show()

## 8. (Preview of Day 7) Project to two dimensions

Each sign lives in 256-dimensional pixel space, which we can't see, so we compress it to two dimensions. Day 7 is about this; here's a first look.

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2).fit_transform(features)
colors = {'cross': 'tab:blue', 'ring': 'tab:orange', 'diag': 'tab:green'}
plt.figure(figsize=(6, 5))
for name in archetypes:
    pts = xy[[i for i, l in enumerate(labels) if l == name]]
    plt.scatter(pts[:, 0], pts[:, 1], label=name, c=colors[name])
plt.legend(); plt.title('Signs projected to 2-D (PCA)'); plt.show()

## 9. The real sign images

> 🔧 **TO BUILD**—replace the made-up shapes with real hieratic sign crops from PaPYrus/Isut. The pipeline doesn't change; only the source of `images` and `labels` does.

In [ ]:
# === TO BUILD: load real sign crops and build `images` + `labels` ===
# from skimage.io import imread
# from skimage.transform import resize
# def load_sign(path):
#     img = imread(path, as_gray=True)
#     return resize(img, (16, 16))
SIGN_IMAGE_FOLDER = None  # a folder of labelled sign crops
# A cleverer feature than raw pixels (stretch): scikit-image's HOG descriptor.
# from skimage.feature import hog

### Talk it over
- Did the clusters in section 7 match the archetypes? Where did a copy land in the wrong group?
- The heatmap and the tree use the same numbers. Which did you find easier to read, and why?
- **Check-in:** is the script itself your variable? What hands, periods, or genres might you compare by shape?